## Importing Libraries

In [1]:
import os
import shutil
import logging
from concurrent.futures import ThreadPoolExecutor, as_completed
from multiprocessing import cpu_count
import pandas as pd
from tqdm import tqdm

## Path Variables

In [2]:
# ---------------- CONFIG ----------------
excel_path = r"C:\Users\chethan.m\Desktop\Sec 138\Section 138 Format.xlsx"  # <-- update
output_folder = r"C:\Users\chethan.m\Desktop\Sec 138\138"
temp_output_folder = r"C:\Users\chethan.m\Desktop\Sec 138\TEMP"
signature_path = r"C:\Users\chethan.m\Desktop\Sec 138\Sign Of Advocate.png"
background_path = r"C:\Users\chethan.m\Desktop\Sec 138\Advocate BG.png"

## Number of Workers (Change Batch Size Only)

In [3]:
MAX_WORKERS = max(1, cpu_count() - 1)
BATCH_SIZE = 6000

## Generation (Never Change)

In [4]:
os.makedirs(output_folder, exist_ok=True)
os.makedirs(temp_output_folder, exist_ok=True)
logging.basicConfig(filename='pdf_generation.log', level=logging.INFO,
                    format='%(asctime)s %(levelname)s %(message)s')
failure_log_path = "failed_rows.csv"

# ---------------- LETTER BODY ----------------
LETTER_BODY = """Ref. No. «Sl_No»						

To,  
«Name_of_Borrower»  

Dear Sir/Madam,  

Ref. : Loan Account No. «Loan_Account_Number»  
Sub : <b>Legal Notice Under Section 25 (1) of The Payment and Settlement Systems Act, 2007</b>  

Under instructions from and on behalf of our clients <b>CapFloat Financial Services Private Limited</b> (Formerly Zen Lefin Private Limited, earlier brand name ‘Capital Float’ and now operating under the brand name “axio”), a registered non-banking financial company, registered under the provisions of the Companies Act, 1956, and registered with the Reserve Bank of India (RBI), having its registered Office at, Gokaldas Platinum, New No.3, Old No.211, Bellary Road, Upper Palace Orchards, Bengaluru – 560080, Karnataka, we issue this notice to you as hereunder;  

1. Our client is engaged in providing financial / Credit facilities through its various loan products for the benefit of its customers. You, the addressee herein above named, have applied for a credit facility / Loan for your financial necessity.  

2. Our client instructs us to state that, at the time of availing the aforesaid loan you have also executed / authorized in Our Client’s favour an Electronic Clearance Debit Mandate/Electronic Direct Debit Mandate (e-NACH/e-DDM), (hereinafter referred as ECS) towards repayment of your equated / periodical monthly instalments (EMI’s) in respect of the aforesaid loan availed.  

3. Our client instructs us to state that, the aforesaid ECS was executed by you towards partial discharge of your debt to our client which is legally enforceable by our Client against you. That, the aforesaid ECS was presented to our client’s bankers on the Repayment Date and Our Client was intimated by their bankers that the aforesaid ECS has been dishonoured. However, our client has advised us to intimate to you that the EMI/s for the following month/s have not been honoured by your banker for the reason “<b>«Reason_for_Dishonour»</b>” and the same came to be intimated to our clients. You have intentionally and with malafide and criminal intention failed to maintain sufficient funds in your bank account that resulted in the dishonor of the ECS mandate by your banker for the <b>«EMI_Month»</b> month.  

4. Please be informed that the dishonor of the ECS mandate is a punishable offence under Section 25 (1) of Payment and Settlement Systems Act 2007 read along with Section 138 of the Negotiable Instruments Act. ECS mandate issued by a borrower towards payment of EMI/s, any such mandate that is dishonoured and in the event the same is not paid within 15 days from the date of receipt of this notice, then such individual/borrower will be liable for prosecution by way of imprisonment up to two years and/or fine up to double the amount of the EMI or with both.  

5. In view of the foregoing, on behalf of our client, we hereby demand and call upon you to pay to our client the said amount of <b>Rs. «CHQ__NACH_Amount»/-</b> being the dues under the said dishonored ECS mandate within 15 days of receipt of this notice, failing which, we have pre-authorised instructions from our client to initiate legal proceedings under the provisions of section 25 (1) of Payment and Settlement Systems Act 2007, read with Section 138 of the Negotiable Instruments Act, for recovery of dues.  

6. This notice is issued without prejudice to our client’s rights to seek remedies with other appropriate courts for recovery of the entire dues outstanding in your loan account with interest, costs and charges etc.  

7. Kindly ignore this notice if the matter is resolved / if the amount demanded above is already paid. Copy of this notice kept in our office for further necessary legal action.  
"""

# ---------------- BACKGROUND FUNCTION ----------------
def add_background(canvas, doc):
    if background_path and os.path.exists(background_path):
        from reportlab.lib.pagesizes import A4
        page_width, page_height = A4
        canvas.saveState()
        canvas.drawImage(background_path, 0, 0, width=page_width, height=page_height,
                         preserveAspectRatio=True, mask='auto')
        canvas.restoreState()
        
def generate_pdf_from_row(row_dict, temp_output_folder, signature_path):
    try:
        from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image
        from reportlab.lib.pagesizes import A4
        from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
        from reportlab.lib.enums import TA_JUSTIFY, TA_LEFT, TA_RIGHT

        styles = getSampleStyleSheet()
        styles.add(ParagraphStyle(name='Justify', alignment=TA_JUSTIFY, leading=14))
        styles.add(ParagraphStyle(name='Left', alignment=TA_LEFT, leading=14))
        styles.add(ParagraphStyle(name='Right', alignment=TA_RIGHT, leading=14))

        # Extract values from Excel row (with cleaned column names)
        sl_no = str(row_dict.get("Sl._No", "")).strip()
        lan = str(row_dict.get("Loan_Account_Number", "")).strip()
        name = str(row_dict.get("Name_of_Borrower", "")).strip()
        emi_month = str(row_dict.get("EMI_Month", "")).strip()
        dishonour_date = str(row_dict.get("CHQ__NACH_dishonour_date", "")).strip()
        nach_amount = str(row_dict.get("CHQ__NACH_Amount", "")).strip()
        reason = str(row_dict.get("Reason_for_Dishonour", "")).strip()
        borrower_number = str(row_dict.get("Borrower_Number", "")).strip()

        # Replace placeholders in LETTER_BODY
        letter_text = LETTER_BODY.replace("«Sl_No»", sl_no)\
                                 .replace("«Loan_Account_Number»", lan)\
                                 .replace("«Name_of_Borrower»", name)\
                                 .replace("«EMI_Month»", emi_month)\
                                 .replace("«CHQ__NACH_Amount»", nach_amount)\
                                 .replace("«Reason_for_Dishonour»", reason)

        filename = f"{lan}.pdf" if lan else f"row_{sl_no}.pdf"
        tmp_path = os.path.join(temp_output_folder, filename)

        doc = SimpleDocTemplate(tmp_path, pagesize=A4,
                                rightMargin=55, leftMargin=55,
                                topMargin=72+12, bottomMargin=72)

        story = []
        for para in letter_text.split("\n\n"):
            if para.strip():
                story.append(Paragraph(para.strip(), styles['Justify']))
                story.append(Spacer(1, 12))

        # Signature image
        if signature_path and os.path.exists(signature_path):
            img = Image(signature_path, width=120, height=50, hAlign='RIGHT')
            story.append(img)
            story.append(Spacer(1, 12))

        story.append(Paragraph("Authorised Signatory", styles['Right']))

        doc.build(story, onFirstPage=add_background, onLaterPages=add_background)

        return True, lan, tmp_path
    except Exception as e:
        return False, str(row_dict.get("Loan_Account_Number", row_dict.get("Sl._No", "unknown"))), str(e)

# ---------------- BATCH RUNNER ----------------
def run_in_batches(df, temp_output_folder, output_folder, signature_path,
                   max_workers=MAX_WORKERS, batch_size=BATCH_SIZE):
    rows = df.to_dict(orient="records")
    failures = []

    for batch_start in range(0, len(rows), batch_size):
        batch = rows[batch_start: batch_start + batch_size]

        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {executor.submit(generate_pdf_from_row, row, temp_output_folder, signature_path): row for row in batch}

            for fut in tqdm(as_completed(futures), total=len(futures),
                            desc=f"Batch {batch_start+1}-{min(batch_start+batch_size,len(rows))}"):
                row = futures[fut]
                try:
                    success, lan_or_id, result = fut.result()
                    if success:
                        final_path = os.path.join(output_folder, os.path.basename(result))
                        shutil.move(result, final_path)
                    else:
                        logging.error(f"Failed {lan_or_id}: {result}")
                        failures.append({**row, "error": result})
                except Exception as e:
                    logging.exception("Unhandled exception in future")
                    failures.append({**row, "error": str(e)})

    if failures:
        pd.DataFrame(failures).to_csv(failure_log_path, index=False)
        print(f"Completed with {len(failures)} failures. See {failure_log_path} and pdf_generation.log")
    else:
        print("Completed successfully with zero failures.")

# ---------------- MAIN ----------------
if __name__ == "__main__":
    df = pd.read_excel(excel_path)
    df.columns = [col.strip().replace(" ", "_").replace("-", "_") for col in df.columns]

    os.makedirs(temp_output_folder, exist_ok=True)
    os.makedirs(output_folder, exist_ok=True)

    run_in_batches(df, temp_output_folder, output_folder, signature_path,
                   max_workers=MAX_WORKERS, batch_size=BATCH_SIZE)


Batch 1-4601: 100%|████████████████████████████████████████████████████████████████| 4601/4601 [10:51<00:00,  7.06it/s]

Completed successfully with zero failures.
